### Model Inspection:
Sneaking into model architecture, target module names for applying  LoRA
Setting up LoRA Config

In [5]:
from transformers import AutoModelForCausalLM

In [6]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(model)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2381.87it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

In [ ]:
# explicitly get module names to avoid assumption
# we'll add lora in projection layers, not everywhere. We decide target modules to attach LoRA
for name, module in model.named_modules():
    if 'proj' in name:
        print(name," ", 'Type: ', type(module))

model.layers.0.self_attn.q_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.k_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.v_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.gate_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.up_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.down_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.q_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.k_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.v_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.o_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.mlp.gate_proj   Type:  <class 'torch.nn.modules.linear.Linear'>
model.layers.1.mlp.up_proj   T

In [12]:
# let's get the linear layers as well
import torch
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

In [ ]:
# LoRA Config Setup
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',    # -> don't train bias parameters in adapter
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        'q_proj',
        'v_proj'
    ]
)


In [15]:
# attaching LoRA to Model
# we'll use get_peft_model() [traditional approach] while we can use add_adapter() directly
from peft import get_peft_model
l_model = get_peft_model(
    model,
    lora_config
)
print(l_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576, padding_idx=2)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [18]:
# now model's total parameters and trainable parameters
total_params = sum(
    p.numel() for p in l_model.parameters()
)
trainable_params = sum(
    p.numel() for p in l_model.parameters() if p.requires_grad
)

print("Total Model Parameters: ",total_params)
print("Total Trainable Model Parameters: ",trainable_params)
print("Total Trainable Model Parameters(%): ",round((trainable_params*100)/total_params,2))



Total Model Parameters:  134975808
Total Trainable Model Parameters:  460800
Total Trainable Model Parameters(%):  0.34


In [20]:
# inspecting exactly what's trainable
for name, param in l_model.named_parameters():
    if param.requires_grad:
        print(name," Shape: " ,param.shape)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight  Shape:  torch.Size([8, 576])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight  Shape:  torch.Size([576, 8])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight  Shape:  torch.Size([8, 576])
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight  Shape:  torch.Size([192, 8])
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight  Shape:  torch.Size([8, 576])
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight  Shape:  torch.Size([576, 8])
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight  Shape:  torch.Size([8, 576])
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight  Shape:  torch.Size([192, 8])
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight  Shape:  torch.Size([8, 576])
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight  Shape:  torch.Size(

In [ ]:
# frozen base and Trainable LoRA Params count
# debugging move
frozen = 0
trainable = 0
for name, param in l_model.named_parameters():
    if param.requires_grad:
        trainable+=param.numel()
    else:
        frozen+=param.numel()

print("Frozen: ", frozen)
print("Trainable LoRA: ", trainable)



Frozen:  134515008
Trainable LoRA:  460800


### Conceptual Importance of Freezing
During full fine tuning, the optimizer needs to deal with gradients and optimizer state
for a huge amount parameters.
with PEFT, we keep the base frozen and update only the trainable parameters

In [ ]:
# dataset format conversion for SFT